In [1]:
import sys
sys.path.append('/home/lytq/Spatial-Transcriptomics-Benchmark/utils')
from sdmbench import compute_ARI, compute_NMI, compute_CHAOS, compute_PAS, compute_ASW, compute_HOM, compute_COM
from load_st_data import load_colon_visium_hd

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/lytq/.conda/envs/SEDR/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_mtx from `anndata` is dep

In [2]:
pred_keys = {
    'STAGATE': 'STAGATE',
}

umap_keys = {
    'GraphST': ['UMAP1', 'UMAP2'],
    'SEDR': ['UMAP1', 'UMAP2'],
    'SpaceFlow': ['UMAP1', 'UMAP2'],
    'STAGATE': ['UMAP1', 'UMAP2'],
    'SpaGCN': ['UMAP1', 'UMAP2'],
    'BayesSpace': ['UMAP1', 'UMAP2'],
    'Seurat': ['UMAP_1', 'UMAP_2'],
    'conST': ['UMAP1', 'UMAP2'],
}

methods = ['STAGATE']

SEEDS = [42, 123, 456, 789, 2024]
dataset = 'colon_cancer'
sample_name = 'visium_hd_cancer_colon_square_016um'

data_folder = f'../../data/{dataset}'
input_dir = f'../../results/{dataset}/'
output_dir = f'../../results/{dataset}_main/'
os.makedirs(output_dir, exist_ok=True)

In [9]:
fontsize = 12

def plot_clustering(adata, method, metrics, out_path, show=False):
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    sc.pl.spatial(
        adata,
        color='pred',
        # spot_size=2,          # important for 16 µm HD bins
        ax=ax,
        show=False,
    )
    ax.set_title(
        f'{method} (ARI={metrics["ARI"]:.4f})',
        fontsize=fontsize,
        fontweight='bold',
    )
    ax.axis('off')
    ax.set_aspect('equal', adjustable='datalim')
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'clustering.pdf'), format='pdf', bbox_inches='tight')
    plt.savefig(os.path.join(out_path, 'clustering.png'), dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close()


def plot_umap(adata, method, out_path, show=False):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    sc.pl.umap(adata, color='gt', ax=axes[0], show=False)
    sc.pl.umap(adata, color='pred', ax=axes[1], show=False)
    axes[0].set_title('Manual Annotation', fontsize=fontsize)
    axes[1].set_title(method, fontsize=fontsize)
    for ax in axes:
        ax.set_aspect('equal', adjustable='datalim')
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, 'umap.pdf'), format='pdf', bbox_inches='tight')
    plt.savefig(os.path.join(out_path, 'umap.png'), dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close()

In [5]:
file_path = os.path.join(data_folder, sample_name)
spatial_dir = os.path.join(file_path, 'spatial')

adata_base, ann_df = load_colon_visium_hd(file_path, spatial_dir)
adata_base.var_names_make_unique()

Loading count matrix ...
Loaded 50387 spots x 18051 genes
Ground Truth
Neoplasm                     20887
Non-neoplastic Epithelium    15207
Connective Tissue            10614
Smooth Muscle                 1713
Vessel                        1044
Veins                          922
Name: count, dtype: int64


In [10]:
skip_index_align = {'SpaceFlow', 'SpaGCN', 'stLearn'}
all_metrics = []

for method in methods:
    pred_key = pred_keys[method]
    umap_key = umap_keys[method]

    print(f'================= Processing {method} {dataset} =================')
    for seed in SEEDS:
        print(f'================== Seed {seed} =================')

        result_dir = os.path.join(input_dir, f'{seed}/{method}')
        if not os.path.isdir(result_dir):
            print(f'  Skipping missing: {result_dir}')
            continue

        out_path = os.path.join(output_dir, f'{seed}/{method}')
        os.makedirs(out_path, exist_ok=True)

        adata = adata_base.copy()

        metadata = pd.read_csv(os.path.join(result_dir, 'cell_metadata.csv'), index_col=0)
        umap_coords = pd.read_csv(os.path.join(result_dir, 'spatial_umap_coords.csv'))
        metrics_saved = pd.read_csv(os.path.join(result_dir, 'metrics.csv'))

        if method not in skip_index_align:
            metadata = metadata.loc[adata.obs_names.intersection(metadata.index)]

        common = adata.obs_names.intersection(metadata.index)
        adata = adata[common].copy()
        metadata = metadata.loc[common]

        adata.obs['gt'] = ann_df.loc[common, 'fine_annot_type'].values

        pred_key_use = pred_key
        if pred_key_use not in metadata.columns:
            alt = pred_key.replace('.', '_')
            if alt in metadata.columns:
                pred_key_use = alt
            else:
                raise KeyError(
                    f'{pred_key} not found in {method} metadata: {metadata.columns.tolist()}'
                )

        pred = pd.to_numeric(metadata[pred_key_use], errors='coerce')
        if pred.isna().any():
            pred = pd.Series(pd.factorize(metadata[pred_key_use])[0], index=metadata.index)

        if pred.min() == 0:
            pred = pred + 1

        adata.obs['pred'] = pred.astype(str)
        umap_idx = umap_coords.set_index('spot_id').loc[common, umap_key]
        adata.obsm['X_umap'] = umap_idx.values

        adata = adata[~pd.isnull(adata.obs['gt'])].copy()
        adata.obs['gt'] = adata.obs['gt'].astype(str)
        adata.obs['pred'] = adata.obs['pred'].astype('category')

        results = {
            'Method': method,
            'SEED': seed,
            'ARI': compute_ARI(adata, 'gt', 'pred'),
            'AMI': compute_NMI(adata, 'gt', 'pred'),
            'Homogeneity': compute_HOM(adata, 'gt', 'pred'),
            'Completeness': compute_COM(adata, 'gt', 'pred'),
            'ASW': compute_ASW(adata, 'pred'),
            'CHAOS': compute_CHAOS(adata, 'pred'),
            'PAS': compute_PAS(adata, 'pred'),
            'Time (s)': float(metrics_saved['Time'].iloc[0]),
            'Memory (MB)': float(metrics_saved['Memory'].iloc[0]),
        }

        pd.DataFrame([results]).to_csv(os.path.join(out_path, 'metrics.csv'), index=False)
        all_metrics.append(results)

        # Draw figures like DLPFC.ipynb
        plot_clustering(adata, method, results, out_path, show=False)
        plot_umap(adata, method, out_path, show=False)

        print(f'  ARI = {results["ARI"]:.4f}')
        print(f'  Saved clustering.pdf / umap.pdf to {out_path}')

    print(f'================= Finished {method} {dataset} =================')

combined = pd.DataFrame(all_metrics).sort_values(['Method', 'SEED'])
combined.to_csv(os.path.join(output_dir, 'colon_cancer_metrics.csv'), index=False)
combined

================= Processing STAGATE colon_cancer =================
================== Seed 42 =================
  ARI = 0.3956
  Saved clustering.pdf / umap.pdf to ../../results/colon_cancer_main/42/STAGATE
================== Seed 123 =================
  ARI = 0.5789
  Saved clustering.pdf / umap.pdf to ../../results/colon_cancer_main/123/STAGATE
================== Seed 456 =================
  ARI = 0.3935
  Saved clustering.pdf / umap.pdf to ../../results/colon_cancer_main/456/STAGATE
================== Seed 789 =================
  ARI = 0.5488
  Saved clustering.pdf / umap.pdf to ../../results/colon_cancer_main/789/STAGATE
================== Seed 2024 =================
  ARI = 0.4739
  Saved clustering.pdf / umap.pdf to ../../results/colon_cancer_main/2024/STAGATE
================= Finished STAGATE colon_cancer =================


,Method,SEED,ARI,AMI,Homogeneity,Completeness,ASW,CHAOS,PAS,Time (s),Memory (MB)
0,STAGATE,42,0.395611,0.453874,0.504903,0.412213,-0.032265,0.013496,0.048068,324.754007,1800.669013
1,STAGATE,123,0.578874,0.589971,0.662539,0.531730,0.013889,0.013500,0.034513,281.268849,1712.383586
2,STAGATE,456,0.393460,0.453925,0.505261,0.412058,-0.031569,0.013492,0.047076,280.323420,1800.670035
3,STAGATE,789,0.548835,0.576325,0.649426,0.518015,0.004393,0.013503,0.038482,290.403877,1715.918462
4,STAGATE,2024,0.473903,0.518161,0.590034,0.461897,0.019222,0.013529,0.040526,226.511388,1716.659492


In [11]:
all_metrics = []
output_path = f'../../results/colon_cancer/colon_cancer_metrics.csv'

for seed in SEEDS:
    print(seed)
    input_dir = f'../../results/colon_cancer/{seed}'
    input_files = glob.glob(input_dir + '/*')
    input_files = [f for f in input_files if os.path.isdir(f)]

    for file in input_files:
        df_metrics = pd.read_csv(os.path.join(file, 'metrics.csv'))
        df_metrics = df_metrics.rename(columns={'Unnamed: 0': 'Method'})
        all_metrics.append(df_metrics)

all_metrics = sorted(
    all_metrics,
    key=lambda x: (x['Method'].iloc[0], x['SEED'].iloc[0])
)

# with open(output_path, 'w') as f:
#     for df in all_metrics:
#         df.to_csv(f, index=False, header=not f.tell())

42
123
456
789
2024


In [13]:
all_metrics

[    Method  SEED       ARI       AMI  Homogeneity  Completeness       ASW  \
 0  GraphST    42  0.329436  0.441874     0.480083      0.409298 -0.062885   
 
       CHAOS       PAS     Time (s)    Memory (MB)  
 0  0.013559  0.064402  1823.673221  120220.660575  ,
     Method  SEED       ARI       AMI  Homogeneity  Completeness       ASW  \
 0  GraphST   123  0.320671  0.478365      0.51283      0.448242 -0.085297   
 
       CHAOS     PAS    Time (s)    Memory (MB)  
 0  0.013563  0.0569  1893.13047  120205.135571  ,
     Method  SEED      ARI       AMI  Homogeneity  Completeness       ASW  \
 0  GraphST   456  0.36302  0.473236     0.503577      0.446344 -0.064097   
 
       CHAOS       PAS     Time (s)    Memory (MB)  
 0  0.013606  0.061901  1859.638353  120220.654097  ,
     Method  SEED       ARI       AMI  Homogeneity  Completeness       ASW  \
 0  GraphST   789  0.347011  0.394448     0.441803      0.356262 -0.002634   
 
       CHAOS       PAS     Time (s)    Memory (MB)  
 0

In [14]:
with open(output_path, 'w') as f:
    for df in all_metrics:
        df.to_csv(f, index=False, header=not f.tell())
    print(f'Combined metrics saved to {output_path}')

Combined metrics saved to ../../results/colon_cancer/colon_cancer_metrics.csv
